In [1]:
%pip install anomalib
%pip install lightning kornia torchmetrics torch FrEIA python-dotenv open-clip-torch einops imgaug

from anomalib.data import MVTec
from anomalib.models import Patchcore, EfficientAd
from anomalib import TaskType
from anomalib.engine import Engine
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.6/486.6 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.4/242.4 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 624.4/624.4 kB 49.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.3/833.3 kB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 890.6/890.6 kB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.0/948.0 kB 59.2 MB/s eta 0:00:00
   ━━━━━━━━

### MVTec Dataset
The MVTec Anomaly Detection (MVTec-AD) dataset is a widely used benchmark for evaluating anomaly detection methods, particularly in industrial quality inspection contexts. It contains high-resolution images across 15 object categories, such as "tile," "leather," "bottle," and "grid," each with a variety of normal (defect-free) and anomalous (defective) samples. Anomalies include visual defects like scratches, dents, or incorrect textures, which make the dataset ideal for testing algorithms on diverse real-world defect types.

#### Example Images (Tile)
|!["Good"](datasets/MVTec/tile/test/good/000.png)|!["Crack"](datasets/MVTec/tile/test/crack/000.png)
|-----|-----|


In [2]:
task = TaskType.DETECTION
categories = ["tile", "leather", "grid", "wood"]

datasets_patchcore = []
datasets_efficientAD = []
for category in categories:
    #Patchcore
    datamodule_patch = MVTec(category=category,task=task)
    datamodule_patch.prepare_data()
    datamodule_patch.setup()
    datasets_patchcore.append(datamodule_patch)
    #EfficientAD
    datamodule_ad = MVTec(category=category, task=task, train_batch_size=1)
    datamodule_ad.prepare_data()
    datamodule_ad.setup()
    datasets_efficientAD.append(datamodule_ad)

callbacks = [
    ModelCheckpoint(
        mode="max",
        monitor="image_AUROC",
    ),
    EarlyStopping(
        monitor="image_AUROC",
        mode="max",
        patience=3,
    ),
]

### Patchcore
PatchCore is an anomaly detection model included in the anomalib library, tailored to identify anomalies in image data by analyzing local patches instead of whole images. This patch-based approach is particularly useful for datasets like MVTec-AD, where defects or anomalies are often subtle and confined to small regions.

#### How PatchCore Works
PatchCore extracts local patches from images, creating a high-dimensional feature representation of each patch. During training, it models these patches’ normal patterns to build a baseline of "normality." When evaluating new data, PatchCore identifies deviations from this normal baseline, flagging patches that look significantly different as potential anomalies.

#### The Role of Coresets
To efficiently manage high-dimensional patch data, PatchCore uses coresets. A coreset is a small, representative subset of the training patches that effectively capture the variability in normal images. Instead of using the full dataset of patches, which could be computationally overwhelming, PatchCore relies on these coresets. This reduces memory usage and speeds up inference without losing essential information on normal patterns. By comparing patches from test images to this representative subset, the model can detect anomalies by identifying patches that diverge from the coreset distribution.

In summary, coresets in PatchCore make the model more efficient, enabling it to focus only on the most informative patches while preserving the ability to accurately identify anomalies.

In [3]:
patchcore_model = Patchcore(pre_trained=True)

engine = Engine(task=task, callbacks=callbacks, pixel_metrics='AUROC', max_epochs=5)

category_level_metric = {}
for category, dataset in zip(categories, datasets_patchcore):
    engine.fit(datamodule=dataset, model=patchcore_model)
    results = engine.test(datamodule=dataset, model=patchcore_model)
    category_level_metric[category] = results[0]['image_AUROC']

model.safetensors:   0%|          | 0.00/276M [00:00<?, ?B/s]

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
F1Score class exists for backwards compatibility. It will be removed in v1.1. Please use BinaryF1Score from torchmetrics instead
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.11/dist-packages/lightning/pytorch/core/optimizer.py:182: `LightningModule.configure_optimizers` returned `None`, this fit will run with no optimizer

  | Name                  | Type                     | Params | Mode 
---------------------------------------------------------------------------
0 | model                 | PatchcoreModel           | 24.9 M | train
1 | _transform            | Compose                  | 0      | train
2 | normalization_metrics | MetricCollection         | 0      | train
3 | image_threshold       | F1AdaptiveThreshold      | 0      | train
4 | pixel_threshold       | F1AdaptiveThreshold      | 0      | train
5 | image_metrics         | AnomalibMetricC

Training: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/lightning/pytorch/loops/optimization/automatic.py:132: `training_step` returned `None`. If this was on purpose, ignore this warning...


Validation: |          | 0/? [00:00<?, ?it/s]



Selecting Coreset Indices.:   0%|          | 0/18032 [00:00<?, ?it/s]

Selecting Coreset Indices.:   0%|          | 24/18032 [00:00<01:15, 236.97it/s]

Selecting Coreset Indices.:   0%|          | 53/18032 [00:00<01:07, 266.62it/s]

Selecting Coreset Indices.:   0%|          | 83/18032 [00:00<01:04, 277.67it/s]

Selecting Coreset Indices.:   1%|          | 113/18032 [00:00<01:03, 283.27it/s]

Selecting Coreset Indices.:   1%|          | 142/18032 [00:00<01:02, 284.07it/s]

Selecting Coreset Indices.:   1%|          | 171/18032 [00:00<01:03, 282.26it/s]

Selecting Coreset Indices.:   1%|          | 200/18032 [00:00<01:03, 279.66it/s]

Selecting Coreset Indices.:   1%|▏         | 229/18032 [00:00<01:03, 282.02it/s]

Selecting Coreset Indices.:   1%|▏         | 259/18032 [00:00<01:02, 285.74it/s]

Selecting Coreset Indices.:   2%|▏         | 288/18032 [00:01<01:02, 285.56it/s]

Selecting Coreset Indices.:   2%|▏         | 318/18032 [00:01<01:01, 287.83it/s]

Selecting Coreset Indices.: 

... (7 outputs left out)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        image_AUROC        │            0.5            │
│       image_F1Score       │    0.8444444537162781     │
└───────────────────────────┴───────────────────────────┘

F1Score class exists for backwards compatibility. It will be removed in v1.1. Please use BinaryF1Score from torchmetrics instead
/usr/local/lib/python3.11/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /notebooks/results/Patchcore/MVTec/tile/v3/weights/lightning exists and is not empty.
/usr/local/lib/python3.11/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /notebooks/results/Patchcore/MVTec/tile/latest/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.11/dist-packages/lightning/pytorch/core/optimizer.py:182: `LightningModule.configure_optimizers` returned `None`, this fit will run with no optimizer

  | Name                  | Type                     | Params | Mode 
---------------------------------------------------------------------------
0 | model                 | PatchcoreModel           | 24.9 M | train
1 | _transform            | Compose      

Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        image_AUROC        │    0.9561403393745422     │
│       image_F1Score       │    0.8633093237876892     │
└───────────────────────────┴───────────────────────────┘

# What Does the AUROC Score Mean and How to Interpret It

The Area Under the Receiver Operating Characteristic (AUROC) score is a performance metric used to evaluate the quality of a binary classification model. It represents the probability that a randomly chosen positive instance (anomaly) is ranked higher by the model than a randomly chosen negative instance (normal).

AUROC Score Range: The AUROC score ranges from 0 to 1:

- A score of 1.0 indicates perfect classification, meaning the model is able to perfectly distinguish between normal and anomalous samples.

- A score of 0.5 indicates a model that performs no better than random chance.

- A score below 0.5 suggests that the model's predictions are worse than random, potentially indicating a significant flaw in the model.

In [4]:
average_auroc = sum(category_level_metric.values()) / len(categories)
category_level_metric['Average'] = average_auroc
#print(category_level_auroc)
for category, auroc in category_level_metric.items():
    print(f"{category}: {auroc}")

tile: 0.9898989200592041
leather: 0.6762908101081848
grid: 0.5
wood: 0.9561403393745422
Average: 0.7805825173854828


These score suggests that the Model works very well on detecting defects in Tiles and Wood indicated by a nearly perfect score
On the other side it performs no better than random chance indicated by the score of 0.5 for the grid category.
For leather the model performs a bit better then on grid, but still far off a from being reliable or having "good" score.

On average this model performs fairly good as it achieves nearly perfect scores on two categorys, but the average might be misleading as it still fails on one category.

### EfficientAD
EfficientAD is a lightweight anomaly detection model designed to achieve high performance with low computational requirements. Unlike PatchCore, which focuses on patch-based anomaly detection, EfficientAD uses a more streamlined feature extraction process, making it suitable for real-time or resource-constrained environments.

#### How EfficientAD Works

EfficientAD leverages efficient neural network architectures to extract features from images and uses these features to distinguish between normal and anomalous regions. The model's workflow can be summarized as follows:

- Feature Extraction: EfficientAD uses a convolutional neural network (CNN) to extract relevant features from input images. These features represent different aspects of the image, such as edges, textures, and shapes, that help the model understand the underlying structure of normal images.

- Feature Embedding: After feature extraction, EfficientAD reduces the dimensionality of the feature space by embedding it into a lower-dimensional representation. This step helps in capturing the essential characteristics of the image while discarding redundant information, making the model faster and more memory-efficient.

- Anomaly Scoring: During inference, EfficientAD compares the features extracted from a test image with those from normal images. It calculates an anomaly score based on how much the test image deviates from the learned distribution of normal features. High anomaly scores indicate a higher likelihood of an anomaly being present.

- Thresholding: Finally, the anomaly score is compared to a predefined threshold to determine whether the image contains an anomaly. If the score exceeds the threshold, the image is flagged as anomalous. Otherwise, it is classified as normal.

This architecture is optimized to reduce memory usage while maintaining high accuracy, making it particularly useful for industrial applications where computational resources may be limited. EfficientAD is also capable of being trained quickly, making it a good choice for scenarios where rapid prototyping and deployment are needed.

In [5]:
efficient_ad_model = EfficientAd()
engine = Engine(task=task, max_epochs=5, callbacks=callbacks)

category_level_metric = {}
for category, dataset in zip(categories, datasets_efficientAD):
    engine.fit(datamodule=dataset, model=efficient_ad_model)  # Train EfficientAD
    results = engine.test(datamodule=dataset, model=efficient_ad_model)  # Test EfficientAD
    category_level_metric[category] = results[0]['image_AUROC']


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
F1Score class exists for backwards compatibility. It will be removed in v1.1. Please use BinaryF1Score from torchmetrics instead
/usr/local/lib/python3.11/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /notebooks/results/Patchcore/MVTec/tile/latest/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                  | Type                     | Params | Mode 
---------------------------------------------------------------------------
0 | model                 | EfficientAdModel         | 8.1 M  | train
1 | _transform            | Compose                  | 0      | train
2 | normalization_metrics | MetricCollection         | 0      | train
3 | image_threshold       | F1AdaptiveThreshold      | 0      | train
4 | pixel_threshold       | F1AdaptiveThreshold      | 0      | train
5 | image_metrics 

Training: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/torchvision/transforms/v2/_deprecated.py:43: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.
  warnings.warn(
Calculate teacher channel mean & std: 100%|██████████| 230/230 [00:03<00:00, 57.89it/s]
/usr/local/lib/python3.11/dist-packages/lightning/pytorch/core/module.py:516: You called `self.log('train_st', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/usr/local/lib/python3.11/dist-packages/lightning/pytorch/utilities/data.py:78: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 0. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
/usr/local/lib/python3.11/dist-packages/lightning/pytorch/core/module.py:516: You called `self.log('train_ae', ..., logger=True)` but have no logger configured

Validation: |          | 0/? [00:00<?, ?it/s]

Calculate Validation Dataset Quantiles: 100%|██████████| 4/4 [00:04<00:00,  1.07s/it]


Validation: |          | 0/? [00:00<?, ?it/s]

... (12 outputs left out)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        image_AUROC        │    0.7568922638893127     │
│       image_F1Score       │    0.8444444537162781     │
└───────────────────────────┴───────────────────────────┘

F1Score class exists for backwards compatibility. It will be removed in v1.1. Please use BinaryF1Score from torchmetrics instead
/usr/local/lib/python3.11/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /notebooks/results/EfficientAd/MVTec/tile/v1/weights/lightning exists and is not empty.
/usr/local/lib/python3.11/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /notebooks/results/Patchcore/MVTec/tile/latest/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                     | Type                     | Params | Mode 
------------------------------------------------------------------------------
0 | model                    | EfficientAdModel         | 8.1 M  | train
1 | _transform               | Compose                  | 0      | train
2 | normalization_metrics    | MetricCollection         | 0      | train
3 | image_threshold          | F1AdaptiveThreshold      |

Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        image_AUROC        │            0.5            │
│       image_F1Score       │    0.8633093237876892     │
└───────────────────────────┴───────────────────────────┘

In [6]:
average_auroc = sum(category_level_metric.values()) / len(categories)
category_level_metric['Average'] = average_auroc
#print(category_level_auroc)
for category, auroc in category_level_metric.items():
    print(f"{category}: {auroc}")


tile: 0.9779943227767944
leather: 0.5
grid: 0.7568922638893127
wood: 0.5
Average: 0.6837216466665268


This model also performs very well on Tiles indicated by the nearly perfect score as well.
In contrast to PatchCore this model performs no better then random on the leather and category.
For the grid category this model achieves a reasonable high score and can be considered fairly reliable.

Overall this model performce worse then Patchcore indicated by the lower average and the fact it fails in two categorys.